In [1]:
# notebooks/01_tax_optimization_demo.ipynb
# Run cells in order. Install deps first: pip install taxopt[notebook]

In [2]:
# Cell 1 — Imports
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import date
from dataclasses import dataclass
from typing import cast, Dict
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from taxopt import (
    Portfolio, TaxLot,
    USCapitalGainsPolicy, LotMethod,
    CvxpyOptimizer, OptimizationInputs,
    LotClose, LongOpen, ShortOpen,
    TaxReport,
)


In [3]:
# Cell 2 — Download price data
TICKERS: list[str] = [
    "NVDA", "AAPL", "MSFT", "AMZN", "GOOGL", "AVGO", "GOOG", "META", "TSLA", "BRK-B",
    "JPM", "LLY", "XOM", "JNJ", "WMT", "V", "MU", "COST", "MA", "NFLX",
    # "ABBV", "CVX", "PLTR", "PG", "HD", "CAT", "AMD", "GE", "BAC", "CSCO",
]
TICKERS: list[str] = [
    "VTI", "SPY", "QQQ", "SCHD", "IJR", "VUG", "VTV", "IVE", "IVW", "VO",
    "VEA", "VWO", "IEFA", "EEM", "VXUS", "EWJ", "EZU", "MCHI", "AGG", "BND",
    "BNDX", "SHY", "TLT", "LQD", "VNQ", "GLD", "DBC", "MLPA", "VGT", "XLV",
]
n: int = len(TICKERS)

_raw = yf.download(TICKERS, start="2020-01-01", end="2025-12-31", auto_adjust=True)
assert _raw is not None

raw: pd.DataFrame = cast(pd.DataFrame, _raw["Close"])[TICKERS].dropna()
returns: pd.DataFrame = cast(pd.DataFrame, raw.pct_change().dropna())

print(f"Price data: {raw.index[0].date()} → {raw.index[-1].date()}, {len(raw)} days")
raw.tail(3)


[*********************100%***********************]  30 of 30 completed


Price data: 2020-01-02 → 2025-12-30, 1507 days


Ticker,VTI,SPY,QQQ,SCHD,IJR,VUG,VTV,IVE,IVW,VO,...,BNDX,SHY,TLT,LQD,VNQ,GLD,DBC,MLPA,VGT,XLV
Date,,,,,,,,,,,,,,,,,,,,,
2025-12-26,339.670013,688.429871,623.890015,27.639999,122.935715,495.059998,192.779999,213.402771,124.883224,294.510010,...,48.210159,82.319321,87.115990,109.853714,88.860001,416.739990,22.700001,47.334946,767.169983,156.050003
2025-12-29,338.390015,685.976562,620.869995,27.620001,122.336662,492.540009,192.589996,213.004227,124.263756,293.529999,...,48.160374,82.359093,87.443642,110.002647,89.040001,398.600006,22.490000,47.403610,763.099976,155.809998
2025-12-30,337.850006,685.138916,619.429993,27.629999,121.497993,491.690002,192.369995,212.775085,124.053940,292.920013,...,48.170334,82.378975,87.235138,109.873573,89.220001,398.890015,22.639999,47.511497,760.890015,155.679993


In [4]:
# Cell 3 — Helper functions
def get_prices(as_of: date) -> dict[str, float]:
    ts = raw.index[raw.index <= pd.Timestamp(as_of)]
    row = raw.iloc[0] if len(ts) == 0 else raw.loc[ts[-1]]
    return {t: float(row[t]) for t in TICKERS}


def compute_market_betas(
    lookback_end: date,
    lookback_days: int = 252,
) -> Dict[str, float]:
    """
    Estimate per-stock market betas vs an equal-weight market index
    built from the current TICKERS universe, over a rolling window.
    """
    end_ts = pd.Timestamp(lookback_end)
    win = returns.loc[end_ts - pd.Timedelta(days=lookback_days * 2) : end_ts].dropna(how="any")
    if len(win) < lookback_days:
        raise ValueError(f"Insufficient history on {lookback_end}: {len(win)} days")

    # Use last 'lookback_days' rows
    win = win.tail(lookback_days)

    # Equal-weight market index returns
    r_m = win.mean(axis=1).to_numpy(dtype=float)  # shape (T,)

    # Centered to avoid intercept leakage
    r_m_centered = r_m - r_m.mean()
    denom = float((r_m_centered ** 2).sum())
    if denom < 1e-10:
        # Degenerate case; return zeros
        return {t: 0.0 for t in TICKERS}

    betas: Dict[str, float] = {}
    for t in TICKERS:
        r_i = win[t].to_numpy(dtype=float)
        r_i_centered = r_i - r_i.mean()
        num = float((r_i_centered * r_m_centered).sum())
        betas[t] = num / denom

    return betas


def get_inputs(
    as_of: date,
    max_turnover: float | None = 0.25,
    lookback_days: int = 252,
) -> OptimizationInputs:
    end  = pd.Timestamp(as_of)
    rets = returns.loc[end - pd.Timedelta(days=lookback_days * 2) : end].dropna(how="any")
    if len(rets) < lookback_days:
        raise ValueError(f"Insufficient history on {as_of}: {len(rets)} days")

    cov: np.ndarray = rets.tail(lookback_days).cov().to_numpy() * 252 + np.eye(n) * 1e-6

    # Alpha: 12-1 month momentum → rank → z-score → beta-neutralize → risk-normalize
    momentum = (1 + rets.iloc[-252:-20]).prod() - 1
    ranked   = momentum.rank().to_numpy(dtype=float)
    w        = (ranked - ranked.mean()) / (ranked.std() + 1e-8)  # standardized signal

    betas_dict = compute_market_betas(as_of, lookback_days=lookback_days)
    b = np.array([betas_dict[t] for t in TICKERS], dtype=float)

    # scalar OLS coefficient a for regression w = a * b + residual
    num = float(w @ b)
    den = float(b @ b) if float(b @ b) > 1e-8 else 1.0
    a   = num / den
    v   = w - a * b    # residual weights, now market-beta-neutral

    v /= np.sqrt(max(float(v @ cov @ v), 1e-8))
    alpha = {t: float((cov @ v)[i]) for i, t in enumerate(TICKERS)}

    return OptimizationInputs(
        alpha=alpha,
        covariance=cov,
        assets=TICKERS,
        prices=get_prices(as_of),
        risk_aversion=2.0,
        tax_aversion=1.0,
        gross_leverage=4.0,     # = L + S (where 130/30 = L/S and L=S*(GL+1)/(GL-1))
        net_exposure=0.0,       # = L - S
        max_weight=0.25,
        max_turnover=max_turnover,
        as_of=as_of,
    )


@dataclass
class TaxLedger:
    """
    Tracks cumulative realized gains/losses over the backtest.
    Assumes the investor has external gains to absorb harvested losses
    immediately — standard assumption in the TLH literature.
    No year-end reset, no carryforward complexity.
    """
    st_realized: float = 0.0
    lt_realized: float = 0.0

    def record(self, report: TaxReport) -> None:
        self.st_realized += report.totals_by_type.get("short_term", 0.0)
        self.lt_realized += report.totals_by_type.get("long_term",  0.0)

    def cumulative_tax_value(self, policy: USCapitalGainsPolicy) -> float:
        """Net tax impact of all realized activity. Negative = net tax saving."""
        return self.st_realized * policy.st_rate + self.lt_realized * policy.lt_rate

    def after_tax_nav(
        self,
        portfolio: Portfolio,
        prices: dict[str, float],
        as_of: date,
        policy: USCapitalGainsPolicy,
    ) -> float:
        """
        Pre-tax NAV
        − DTL on unrealized gains (hypothetical liquidation tax today)
        − cumulative tax on realized gains (positive = owed, negative = saved)
        """
        unreal_st, unreal_lt = 0.0, 0.0
        for asset, lots in portfolio.lots.items():
            px = prices[asset]
            for lot in lots:
                if lot.quantity <= 0:
                    continue
                gain = (px - lot.cost_basis) * lot.quantity
                days = (as_of - lot.acquisition_date).days
                if days >= policy.lt_threshold_days:
                    unreal_lt += gain
                else:
                    unreal_st += gain

        dtl = unreal_st * policy.st_rate + unreal_lt * policy.lt_rate
        return portfolio.total_value(prices) - dtl - self.cumulative_tax_value(policy)

In [ ]:
# Cell 4 — rebalance loop with ledger
POLICY = USCapitalGainsPolicy(lot_method=LotMethod.MIN_GAIN)
SOLVER = CvxpyOptimizer(
    solver="SCIP",
    verbose=False,
    tax_aware=True,                 # enable tax-aware optimization (enable objective term)
    relax_turnover=True,            # gradually widens turnover if infeasible
    turnover_relax_step=0.05,       # +5pp per attempt
    turnover_relax_max_attempts=5,  # up to +25pp before giving up
    mip_gap=0.01,                   # 1% MIP gap for faster solve at the cost of optimality guarantee
)

# All month-ends in range; rebalance on each except the last,
# which serves only as the EOM date of the final holding period.
all_dates: list[date] = pd.date_range("2022-12-30", "2025-12-31", freq="ME").date.tolist()
rebal_dates = all_dates[:-1]   # rebalance dates: Jan 2023 … Nov 2025
eom_dates   = all_dates[1:]    # EOM dates:       Feb 2023 … Dec 2025

portfolio = Portfolio(cash=100_000.0)
ledger    = TaxLedger()

# Synthetic first row: initial cash at the first rebal date (before any trades)
history: list[dict] = [{
    "date":          rebal_dates[0],
    "nav_eom":       portfolio.cash,
    "after_tax_nav": portfolio.cash,
    "st_realized":   0.0,
    "lt_realized":   0.0,
    "st_cumulative": 0.0,
    "lt_cumulative": 0.0,
    "tax_alpha":     0.0,
    "num_actions":   0,
    "num_lots":      0,
}]

for i, (rebal_date, eom_date) in enumerate(zip(rebal_dates, eom_dates)):
    is_first   = (i == 0)

    # --- Prices and NAV at the start of the period (BOM / previous EOM) ---
    prices_bom = get_prices(rebal_date)
    tv_pre     = portfolio.total_value(prices_bom)

    # --- Optimize and apply trades at current month-end prices ---
    inputs = get_inputs(rebal_date, max_turnover=None if is_first else None)
    result = SOLVER.solve(portfolio, inputs, POLICY, tv_pre)

    if result.status not in ("optimal", "optimal_inaccurate"):
        print(f"{rebal_date}: {result.status}, skipping")
        continue

    portfolio, tax_report = portfolio.apply_actions(result.actions, prices_bom, POLICY, rebal_date)
    tv_post = portfolio.total_value(prices_bom)
    assert abs(tv_post - tv_pre) / tv_pre < 1e-4, f"NAV not conserved: {tv_pre:.2f} → {tv_post:.2f}"

    # --- Tax alpha this period: tax saving as fraction of NAV ---
    period_tax_impact = (
        tax_report.totals_by_type.get("short_term", 0.0) * POLICY.st_rate +
        tax_report.totals_by_type.get("long_term",  0.0) * POLICY.lt_rate
    )
    tax_alpha = -period_tax_impact / tv_pre

    # --- EOM of this holding period ---
    prices_eom = get_prices(eom_date)
    tv_eom     = portfolio.total_value(prices_eom)

    # --- Record realized gains into ledger AFTER computing period tax_alpha ---
    ledger.record(tax_report)
    at_nav_eom = ledger.after_tax_nav(portfolio, prices_eom, eom_date, POLICY)

    # --- Append row indexed by the EOM date ---
    history.append({
        "date":          eom_date,
        "nav_eom":       tv_eom,
        "after_tax_nav": at_nav_eom,
        "st_realized":   tax_report.totals_by_type.get("short_term", 0.0),
        "lt_realized":   tax_report.totals_by_type.get("long_term",  0.0),
        "st_cumulative": ledger.st_realized,
        "lt_cumulative": ledger.lt_realized,
        "tax_alpha":     tax_alpha,
        "num_actions":   len(result.actions),
        "num_lots":      sum(len(v) for v in portfolio.lots.values()),
    })

    turnover_note = (
        f" [turnover relaxed to {result.effective_turnover:.0%}]"
        if result.effective_turnover != inputs.max_turnover
        else ""
    )
    print(
        f"{eom_date} EOM=${tv_eom:>10,.0f} AT-NAV=${at_nav_eom:>10,.0f} "
        f"ST={tax_report.totals_by_type.get('short_term', 0.0):>+8,.0f} "
        f"LT={tax_report.totals_by_type.get('long_term',  0.0):>+8,.0f} "
        f"TaxAlpha={tax_alpha*100:>+6.3f}%{turnover_note}"
    )

df = pd.DataFrame(history).set_index("date")


2023-01-31 EOM=$    96,459 AT-NAV=$    92,876 ST=      +0 LT=      +0 TaxAlpha=-0.000%
2023-02-28 EOM=$    97,539 AT-NAV=$   100,880 ST= -13,223 LT=      +0 TaxAlpha=+4.798%
2023-03-31 EOM=$    93,233 AT-NAV=$    95,483 ST=  -1,078 LT=      +0 TaxAlpha=+0.387%
2023-04-30 EOM=$    94,917 AT-NAV=$    97,011 ST=  -1,776 LT=      +0 TaxAlpha=+0.667%
2023-05-31 EOM=$    92,791 AT-NAV=$    97,303 ST=    -794 LT=      +0 TaxAlpha=+0.293%
2023-06-30 EOM=$    91,077 AT-NAV=$    93,475 ST=  -3,083 LT=      +0 TaxAlpha=+1.163%
2023-07-31 EOM=$    90,681 AT-NAV=$    92,574 ST=  -3,337 LT=      +0 TaxAlpha=+1.282%
2023-08-31 EOM=$    93,172 AT-NAV=$    98,346 ST=  -4,725 LT=      +0 TaxAlpha=+1.824%
2023-09-30 EOM=$    94,932 AT-NAV=$   102,505 ST=      -0 LT=      +0 TaxAlpha=+0.000%
2023-10-31 EOM=$    97,960 AT-NAV=$   106,571 ST=    -613 LT=      +0 TaxAlpha=+0.226%
2023-11-30 EOM=$    96,817 AT-NAV=$   100,904 ST=  -2,258 LT=      +0 TaxAlpha=+0.807%
2023-12-31 EOM=$    92,298 AT-NAV=$    94,8

In [6]:
# Cell 5 — Summary metrics
final_nav       = float(df["after_tax_nav"].iloc[-1])
pretax_nav      = float(df["nav_eom"].iloc[-1])
initial_nav     = 100_000.0
total_st        = float(df["st_realized"].sum())
total_lt        = float(df["lt_realized"].sum())
total_realized  = total_st + total_lt
cum_tax         = ledger.cumulative_tax_value(POLICY)
pretax_return   = (pretax_nav  / initial_nav - 1) * 100
aftertax_return = (final_nav   / initial_nav - 1) * 100
tax_drag        = pretax_return - aftertax_return
tax_efficiency  = aftertax_return / pretax_return if pretax_return != 0 else float("nan")
tax_alpha_ann   = float(np.prod(df["tax_alpha"].to_numpy(dtype=float) + 1)) ** (12 / len(df)) - 1
n_years         = len(rebal_dates) / 12

print("=" * 52)
print(f"  Backtest: {rebal_dates[0]} → {rebal_dates[-1]}  ({n_years:.1f} yrs)")
print("=" * 52)
print(f"  {'Initial NAV:':<32} ${initial_nav:>10,.0f}")
print(f"  {'Pre-tax final NAV:':<32} ${pretax_nav:>10,.0f}")
print(f"  {'After-tax final NAV:':<32} ${final_nav:>10,.0f}")
print()
print(f"  {'Pre-tax return:':<32} {pretax_return:>10.2f}%")
print(f"  {'After-tax return:':<32} {aftertax_return:>10.2f}%")
print(f"  {'Tax drag:':<32} {tax_drag:>10.2f}%")
print(f"  {'Tax efficiency ratio:':<32} {tax_efficiency:>10.2%}")
print(f"  {'Annualized tax alpha:':<32} {tax_alpha_ann*100:>10.3f}%")
print()
print(f"  {'Cumul. ST realized:':<32} ${total_st:>10,.0f}")
print(f"  {'Cumul. LT realized:':<32} ${total_lt:>10,.0f}")
print(f"  {'Cumul. net realized:':<32} ${total_realized:>10,.0f}")
print(f"  {'Net tax impact (realized):':<32} ${cum_tax:>10,.0f}")
print(f"  {'  (neg = net tax saving)'}")
print()
print(f"  {'Effective rate on realized:':<32} {cum_tax/max(abs(total_realized),1)*100:>10.2f}%")
print(f"  {'Avg actions per rebalance:':<32} {df['num_actions'].mean():>10.1f}")
print(f"  {'Final lot count:':<32} {int(df['num_lots'].iloc[-1]):>10d}")
print("=" * 52)


  Backtest: 2022-12-31 → 2025-11-30  (3.0 yrs)
  Initial NAV:                     $   100,000
  Pre-tax final NAV:               $   105,936
  After-tax final NAV:             $   117,060

  Pre-tax return:                        5.94%
  After-tax return:                     17.06%
  Tax drag:                            -11.12%
  Tax efficiency ratio:               287.39%
  Annualized tax alpha:                 9.603%

  Cumul. ST realized:              $   -79,976
  Cumul. LT realized:              $       872
  Cumul. net realized:             $   -79,104
  Net tax impact (realized):       $   -27,817
    (neg = net tax saving)

  Effective rate on realized:          -35.17%
  Avg actions per rebalance:            114.2
  Final lot count:                        239


In [7]:
# Cell 6 — Interactive charts
dates = df.index.astype(str).tolist()

# ── Chart 1: NAV vs After-Tax NAV ──────────────────────────────────────────
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=dates, y=df["nav_eom"], name="Pre-Tax NAV",
                          line=dict(color="#3498db"), mode="lines+markers",
                          hovertemplate="%{x}<br>Pre-Tax NAV: $%{y:,.0f}<extra></extra>"))
fig1.add_trace(go.Scatter(x=dates, y=df["after_tax_nav"], name="After-Tax NAV",
                          line=dict(color="#2ecc71"), mode="lines+markers",
                          hovertemplate="%{x}<br>After-Tax NAV: $%{y:,.0f}<extra></extra>"))
fig1.update_layout(title="NAV vs After-Tax NAV (EOM)", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="$ Value")
fig1.show()

# ── Chart 2: Realized ST / LT Gains per Rebalance ─────────────────────────
fig2 = go.Figure()
fig2.add_trace(go.Bar(x=dates, y=df["st_realized"], name="ST Realized",
                      marker_color="#e74c3c",
                      hovertemplate="%{x}<br>ST: $%{y:,.0f}<extra></extra>"))
fig2.add_trace(go.Bar(x=dates, y=df["lt_realized"], name="LT Realized",
                      marker_color="#2ecc71",
                      hovertemplate="%{x}<br>LT: $%{y:,.0f}<extra></extra>"))
fig2.update_layout(title="Realized ST / LT Gains per Rebalance",
                   barmode="group", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="$ Gain / Loss")
fig2.show()

# ── Chart 3: Cumulative Realized ST / LT ──────────────────────────────────
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=dates, y=df["st_cumulative"], name="ST Cumulative",
                          line=dict(color="#e74c3c"), mode="lines+markers",
                          hovertemplate="%{x}<br>ST Cumul: $%{y:,.0f}<extra></extra>"))
fig3.add_trace(go.Scatter(x=dates, y=df["lt_cumulative"], name="LT Cumulative",
                          line=dict(color="#2ecc71"), mode="lines+markers",
                          hovertemplate="%{x}<br>LT Cumul: $%{y:,.0f}<extra></extra>"))
fig3.update_layout(title="Cumulative Realized ST / LT Gains", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="$ Cumulative")
fig3.show()

# ── Chart 4: Cumulative Tax Alpha ──────────────────────────────────────────
cum_alpha = df["tax_alpha"].astype(float).cumsum() * 100
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=dates, y=cum_alpha, name="Cumul. Tax Alpha",
                          line=dict(color="#9b59b6"), mode="lines+markers",
                          fill="tozeroy", fillcolor="rgba(155,89,182,0.15)",
                          hovertemplate="%{x}<br>Tax Alpha: %{y:.3f}%<extra></extra>"))
fig4.add_hline(y=0, line_dash="dash", line_color="gray")
fig4.update_layout(title="Cumulative Tax Alpha (% of NAV)", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="% of NAV")
fig4.show()


In [8]:
# Cell 7 — Lot inspection
final_date   = rebal_dates[-1]
final_prices = get_prices(final_date)

rows = [
    {
        "asset":       asset,
        "qty":         round(lot.quantity, 4),
        "basis":       round(lot.cost_basis, 2),
        "price":       round(final_prices[asset], 2),
        "unreal_gain": round((final_prices[asset] - lot.cost_basis) * lot.quantity, 2),
        "days_held":   (final_date - lot.acquisition_date).days,
        "gain_type": POLICY.classify_gain(lot, final_date, 0.0),
    }
    for asset, lots in portfolio.lots.items()
    for lot in lots
]
lot_df = pd.DataFrame(rows).sort_values("unreal_gain")
print(f"Total unrealized:  ${float(lot_df['unreal_gain'].sum()):,.0f}")
print(f"ST unrealized:     ${float(lot_df.loc[lot_df['gain_type']=='ST','unreal_gain'].sum()):,.0f}")
print(f"LT unrealized:     ${float(lot_df.loc[lot_df['gain_type']=='LT','unreal_gain'].sum()):,.0f}")
lot_df


Total unrealized:  $82,219
ST unrealized:     $0
LT unrealized:     $0


,asset,qty,basis,price,unreal_gain,days_held,gain_type
54,VO,-88.5848,289.77,291.01,-109.41,30,short_term
9,SCHD,-268.3839,27.15,27.31,-43.73,244,short_term
155,VTI,-32.5738,334.47,335.36,-28.91,30,short_term
5,QQQ,0.0000,628.26,618.45,-0.00,30,short_term
6,QQQ,-14.2196,618.45,618.45,-0.00,0,short_term
...,...,...,...,...,...,...,...
191,VWO,484.0776,34.97,53.25,8848.93,761,long_term
40,IVE,121.6505,136.70,210.65,8996.29,1065,long_term
71,EEM,481.3613,34.35,53.55,9244.42,761,long_term
2,QQQ,38.1096,342.36,618.45,10521.96,914,long_term
